In [1]:
import importlib.resources
import sys
import os
import json
import numpy as np
import pandas as pd
import pyomo.environ as pyo

In [2]:
root_path = os.path.join(os.getcwd(), "..", "Data", "fossil_results")
# read generator parameters and names
gen_path = gen_path = os.path.join(os.getcwd(), "..", "Data", "gen_dict.json")
with open(gen_path, 'rb') as f:
    gen_dict = json.load(f)
gen_names = list(gen_dict["fossil"].keys())

In [3]:
# check the result of one generator.
gen_csv_path = os.path.join(root_path, "gen_" + gen_names[0] + "_result.csv")
df_gen = pd.read_csv(gen_csv_path)
df_gen

,Time,LMP,power_to_grid,gen_101_CT_1.op_mode,gen_101_CT_1.startup,gen_101_CT_1.shutdown,gen_101_CT_1.power,elec_revenue,total_hourly_cost,total_hourly_revenue,hourly_startup_cost,net_hourly_cash_inflow,gen_101_CT_1.vom
0,1,0.000000,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.000000,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,8780,26.324557,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8780,8781,26.324557,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8781,8782,76.324557,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8782,8783,75.908700,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# get column_info
def get_col_ts_info(df, col_name):
    return df[col_name]

def summarize_gen_pt_result(df, gen_name):
    tot_dispatch = get_col_ts_info(df_gen, "power_to_grid").sum()
    tot_elec_rev = get_col_ts_info(df_gen, "elec_revenue").sum()
    tot_op_cost = get_col_ts_info(df_gen, "total_hourly_cost").sum()
    tot_startup_cost = get_col_ts_info(df_gen, "hourly_startup_cost").sum()
    tot_profit = get_col_ts_info(df_gen, "net_hourly_cash_inflow").sum()
    num_startup = get_col_ts_info(df_gen, f"gen_{gen_name}.startup").sum()
    
    sum_dict = {"tot_dispatch": [tot_dispatch],
                "tot_elec_rev": [tot_elec_rev],
               "tot_op_cost": [tot_op_cost],
               "tot_startup_cost": [tot_startup_cost],
               "tot_profit": [tot_profit],
               "num_startup": [int(num_startup)],}
    df_summarize = pd.DataFrame(sum_dict)
    df_summarize.index = [gen_name]
    
    return df_summarize

summarize_gen_pt_result(df_gen, gen_names[0])

,tot_dispatch,tot_elec_rev,tot_op_cost,tot_startup_cost,tot_profit,num_startup
101_CT_1,3316.0,4.407069e+06,394224.227869,2328.615,4.010517e+06,45


In [6]:
df_all_gen_summarize = pd.DataFrame()
for name in gen_names:
    gen_csv_path = os.path.join(root_path, "gen_" + name + "_result.csv")
    df_gen = pd.read_csv(gen_csv_path)
    df_summarize = summarize_gen_pt_result(df_gen, name)
    df_all_gen_summarize = pd.concat([df_all_gen_summarize, df_summarize], ignore_index=False)

In [7]:
df_all_gen_summarize

,tot_dispatch,tot_elec_rev,tot_op_cost,tot_startup_cost,tot_profit,num_startup
101_CT_1,3316.0,4.407069e+06,3.942242e+05,2.328615e+03,4.010517e+06,45
101_CT_2,3316.0,4.407069e+06,3.942242e+05,2.328615e+03,4.010517e+06,45
101_STEAM_3,389772.0,2.800817e+07,8.271056e+06,6.197809e+05,1.911733e+07,58
101_STEAM_4,389772.0,2.800817e+07,8.271056e+06,6.197809e+05,1.911733e+07,58
102_CT_1,3268.0,4.401271e+06,4.057930e+05,2.276868e+03,3.993202e+06,44
...,...,...,...,...,...,...
321_CC_1,554435.0,1.037976e+08,1.509489e+07,2.916855e+06,8.578583e+07,104
322_CT_5,53284.0,1.507211e+07,1.925696e+06,5.608582e+05,1.258556e+07,99
322_CT_6,53284.0,1.507211e+07,1.925696e+06,5.608582e+05,1.258556e+07,99
323_CC_1,534155.0,1.043119e+08,1.528252e+07,3.141228e+06,8.588816e+07,112


## Read the Prescient Results

In [27]:
dispatch_path = "Generator_Dispatch.csv"
lmp_path = "Bus_LMP.csv"

df_lmp = pd.read_csv(lmp_path)
df_dispatch = pd.read_csv(dispatch_path)

print(df_lmp.columns)
print(df_dispatch.columns)

Index(['Datetime', 'Abel_LMP', 'Abel_LMP DA', 'Adams_LMP', 'Adams_LMP DA',
       'Alder_LMP', 'Alder_LMP DA', 'Arne_LMP', 'Arne_LMP DA', 'Arthur_LMP',
       'Arthur_LMP DA', 'Asser_LMP', 'Asser_LMP DA', 'Astor_LMP',
       'Astor_LMP DA', 'Austen_LMP', 'Austen_LMP DA', 'Bach_LMP',
       'Bach_LMP DA', 'Bacon_LMP', 'Bacon_LMP DA', 'Baker_LMP', 'Baker_LMP DA',
       'Barlow_LMP', 'Barlow_LMP DA', 'Barton_LMP', 'Barton_LMP DA',
       'Basov_LMP', 'Basov_LMP DA', 'Bayle_LMP', 'Bayle_LMP DA', 'Behring_LMP',
       'Behring_LMP DA', 'Bloch_LMP', 'Bloch_LMP DA', 'Cabell_LMP',
       'Cabell_LMP DA', 'Cabot_LMP', 'Cabot_LMP DA', 'Carew_LMP',
       'Carew_LMP DA', 'Cecil_LMP', 'Cecil_LMP DA', 'Chase_LMP',
       'Chase_LMP DA', 'Chifa_LMP', 'Chifa_LMP DA', 'Clark_LMP',
       'Clark_LMP DA', 'Cobb_LMP', 'Cobb_LMP DA', 'Cole_LMP', 'Cole_LMP DA',
       'Comte_LMP', 'Comte_LMP DA'],
      dtype='object')
Index(['Datetime', '101_CT_1_Dispatch', '101_CT_1_Dispatch DA',
       '101_CT_2_Dispat

### Calculate the total dispatch

In [25]:
# calculate the total dispatch
gen_names = list(gen_dict["fossil"].keys())
dispatch_result = {}

for name in gen_names:
    # calculate annual DA/RT dispatch
    dispatch_result[name] = {}
    dispatch_result[name]["tot_Dispatch_DA"] = df_dispatch[name+"_Dispatch DA"].sum()
    dispatch_result[name]["tot_Dispatch"] = df_dispatch[name+"_Dispatch"].sum()

# dispatch_result

### Summarize the LMP infomation

In [31]:
LMP_result = {}

for name in gen_names:
    bus_name = gen_dict["fossil"][name]["bus_name"]
    LMP_result[bus_name] = {}
    LMP_result[bus_name][f"LMP_DA_mean"] = df_lmp[f"{bus_name}_LMP DA"].mean()
    LMP_result[bus_name][f"LMP_DA_median"] = df_lmp[f"{bus_name}_LMP DA"].median()
    LMP_result[bus_name][f"LMP_DA_min"] = df_lmp[f"{bus_name}_LMP DA"].min()
    LMP_result[bus_name][f"LMP_DA_max"] = df_lmp[f"{bus_name}_LMP DA"].max()
    LMP_result[bus_name][f"LMP_mean"] = df_lmp[f"{bus_name}_LMP"].mean()
    LMP_result[bus_name][f"LMP_median"] = df_lmp[f"{bus_name}_LMP"].median()
    LMP_result[bus_name][f"LMP_min"] = df_lmp[f"{bus_name}_LMP"].min()
    LMP_result[bus_name][f"LMP_max"] = df_lmp[f"{bus_name}_LMP"].max()
    
# LMP_result

## Compare PCM with PT